In [ ]:
# pip install arxiv

# Chatbot Example

## Import Libraries

In [ ]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
from gates_openai import create_response

## Tool Functions

In [ ]:
PAPER_DIR = "papers"

The first tool searches for relevant arXiv papers based on a topic and stores the papers' info in a JSON file (title, authors, summary, paper url and the publication date). The JSON files are organized by topics in the `papers` directory. The tool does not download the papers.  

In [ ]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.
    
    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)
        
    Returns:
        List of paper IDs found in the search
    """
    
    # Use arxiv to find the papers 
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    # Create directory for this topic
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")

    # Try to load existing papers info
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    # Process each paper and add to papers_info  
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    # Save updated papers_info to json file
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    
    return paper_ids

In [ ]:
# Sanity Check
search_papers("computers")

The second tool looks for information about a specific paper across all topic directories inside the `papers` directory.

In [ ]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """
 
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."

In [ ]:
# Sanity Check
print(extract_info('1312.3300v1'))

## Tool Schema

Here are the schema of each tool which you will provide to the LLM.

In [ ]:
tools = [
    {
        "type": "function",
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "parameters": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "The topic to search for"
                }, 
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to retrieve",
                    "default": 5
                }
            },
            "required": ["topic"]
        }
    },
    {
        "type": "function",
        "name": "extract_info",
        "description": "Search for information about a specific paper downloaded using search_papers across all topic directories.",
        "parameters": {
            "type": "object",
            "properties": {
                "paper_id": {
                    "type": "string",
                    "description": "The ID of the paper to look for"
                }
            },
            "required": ["paper_id"]
        }
    }
]

## Tool Mapping

This code handles tool mapping

In [ ]:
TOOL_MAPPING = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

## Chatbot Code

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

### EXERCISE: Query Processing

### Instructions
- Create a system prompt and complete the message list
- Create a kwargs key  `previous_response_id` and pass the argument previous_response_id as value
- Add the model `gpt-4o-mini`, the list of tools and messages, and unpack kwargs inside the create_response method
- List all the function calls from the response object
- Retrun output text and response id if no function calls found

***
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS:</h4>

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment
 
---

In [ ]:
def process_query(query: str = None, previous_response_id: str = None):
    
    ### START CODE HERE ###

    # Create a system prompt for the research assistant
    # It should use the search_papers tool to search arxiv for academic papers on a given topic
    # It should use extract_info tool to get information on papers retrieved throught the search_papers tool
    # Pass in the query as user message
    messages = [
        {
            "role": "system",
            "content": None
        },
        {
            "role":"user",
            "content":None
        }
    ]
    
    # In OpenAI responses API, previous_response_id is used to chain the responses.
    # However we can't pass a Null value specially on the first turn.
    # We need to create kwargs to pass it in the create_response proxy only if there is a previous_response_id
    kwargs = {}

    # Create a kwargs key  `previous_response_id` and pass the argument previous_response_id as value
    if previous_response_id:
        pass # Replace this with actual key-value assignment
    
    # Add the model `gpt-4o-mini`, the list of tools and messages, and unpack kwargs inside
    response = create_response(
        model = None,
        tools = None,
        input = None,
        # Unpack kwargs here
    )

    # List all the function calls from the response object
    function_calls = []

    # Retrun output text and response id if no function calls found
    if not function_calls:
        return None

    ### END CODE HERE ###
    
    while True:

        tool_outputs = []

        for call in function_calls:
            args = json.loads(call.arguments)
            result = TOOL_MAPPING[call.name](**args)

            tool_output = "\n".join(map(str, result))
            # print(tool_output)

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": tool_output
            })

        messages.extend(tool_outputs)

        response = create_response(
            model = "gpt-4o-mini",
            input = tool_outputs,
            tools = tools,
            previous_response_id = response.id
        )


        function_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        if function_calls:
            print("Function calls found...")
            continue
        else:
            return response.output_text, response.id



### Chat Loop

In [ ]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    response_id = None
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            response, response_id = process_query(query, response_id)
            print("\n")
            print(response)
        except Exception as e:
            print(f"\nError: {str(e)}")

Feel free to interact with the chatbot. Here's an example query: 

- Search for 2 papers on "LLM interpretability"

To manually access the `papers` folder: look for it in Workshop/MCP/1-ChatBot

In [ ]:
chat_loop()

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI chat models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from others.</p>